In [1]:
import pandas as pd
import numpy as np
import seaborn as sb
from matplotlib import pyplot as plt
%matplotlib inline

In [2]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [4]:
for c in df.columns:
    filler = 0
    if df[c].dtype == 'str':
        df[c] = df[c].str.lower().str.replace(' ', '_')
        filler = 'NA'
    df[c] = df[c].fillna(filler)

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,europe,gasoline,front-wheel_drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,europe,diesel,front-wheel_drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,asia,gasoline,front-wheel_drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,europe,gasoline,front-wheel_drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,usa,diesel,front-wheel_drive,3,2580,7,304.0,4510,17.5,31.0


In [5]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_full_train.reset_index(drop=True)
df_test.reset_index(drop=True)
df_train.reset_index(drop=True)
df_val.reset_index(drop=True)

y_test = df_test['fuel_efficiency_mpg'].values
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

del df_test['fuel_efficiency_mpg']
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']

In [6]:
from sklearn.feature_extraction import DictVectorizer

train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

In [7]:
# Question one

from sklearn.tree import DecisionTreeRegressor, export_text

dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [8]:
# Question two

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
root_mean_squared_error(y_val, y_pred)

1.8332197358745623

In [9]:
# Question three

for est in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=est, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    print(f'N_Estimators: {est} -> RMSE: {root_mean_squared_error(y_val, y_pred)}')

N_Estimators: 10 -> RMSE: 1.8332197358745623
N_Estimators: 50 -> RMSE: 1.7739880309630047
N_Estimators: 100 -> RMSE: 1.7697862961386042
N_Estimators: 150 -> RMSE: 1.769198235673751


In [10]:
# Question four

for d in [10, 15, 20, 25]:
    rmse = []
    for est in [10, 50, 100, 150]:
        rf = RandomForestRegressor(max_depth=d, n_estimators=est, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
    
        y_pred = rf.predict(X_val)
        rmse.append(root_mean_squared_error(y_val, y_pred))
    print(f'Mean RMSE with depth={d} -> {np.array(rmse).mean().round(3)}')


Mean RMSE with depth=10 -> 1.754
Mean RMSE with depth=15 -> 1.784
Mean RMSE with depth=20 -> 1.787
Mean RMSE with depth=25 -> 1.786


In [20]:
# Question five

rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

df_importance = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_
}).sort_values(by='importance', ascending=False)
print(df_importance)

                         feature  importance
9                     model_year    0.296107
15                vehicle_weight    0.203313
6             fuel_type=gasoline    0.160712
8                     horsepower    0.078271
4            engine_displacement    0.061281
0                   acceleration    0.054998
14                    origin=usa    0.042152
2   drivetrain=front-wheel_drive    0.020062
11                     num_doors    0.019401
5               fuel_type=diesel    0.016571
7               fuel_type=hybrid    0.016272
1     drivetrain=all-wheel_drive    0.007951
10                 num_cylinders    0.007318
12                   origin=asia    0.006515
3    drivetrain=rear-wheel_drive    0.005638
13                 origin=europe    0.003436


In [23]:
# Question six
import xgboost as xgb

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

for eta in [0.1, 0.3]:
    xgb_params = {
        'eta': eta, 
        'max_depth': 6,
        'min_child_weight': 1,
        
        'objective': 'reg:squarederror',
        'nthread': 8,
        
        'seed': 1,
        'verbosity': 1,
    }

    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                    verbose_eval=5)

    y_pred = model.predict(dval)
    print(f'RMSE {round(root_mean_squared_error(y_val, y_pred), 3)} con eta={eta}')


RMSE 1.725 con eta=0.1
RMSE 1.826 con eta=0.3
